In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import pymc as pm
import arviz as az
import matplotlib
import time as time

from scipy.interpolate import CubicSpline

from sensor_ml.util.Plotting import *

In [ ]:
nai_distribution = np.load('Spectra/NaI_Distribution.npy')
print(nai_distribution.shape)

nai_spectrum_data = np.load('Spectra/NaI_Response.npy')
print(nai_spectrum_data.shape)
bins = np.concatenate((nai_spectrum_data[:,0].ravel(), np.array([40.5]))) # Hacky
nai_spectrum_response = nai_spectrum_data[:,1]

cmap_color = 'plasma'
shading = 'auto'
binWidths = (bins - np.roll(bins, 1))[1:]
binCenters = (bins + np.roll(bins, 1))[1:]/2

def plot_distribution(data_, bins_, binWidths_):
    plt.figure()
    plt.pcolormesh(bins_, bins_, (data_/binWidths_).T,
                   shading=shading,cmap=cmap_color, norm=LogNorm(vmin=1E-3, vmax=1E0))
    cb = plt.colorbar()
    cb.ax.tick_params(labelsize=16)
    cb.set_label(label='Truncated Probability Density',size=18)
    plt.xlim(.1,35)
    plt.ylim(.1,35)
    plt.xscale('log')
    plt.yscale('log')
    plt.title('NaI Response Matrix',fontsize=20)
    plt.xlabel('True Energy (MeV)',fontsize=18)
    plt.ylabel('Measured Energy (MeV)',fontsize=18)
    plt.tick_params(labelsize=16)
    
plot_distribution(nai_distribution, bins, binWidths)

print(np.sum(nai_distribution[:,0]), np.sum(nai_distribution[0, :])) # [True, Measured]


fig, axes = plt.subplots(1, 3, figsize=(12,3), dpi=200)
n = 20
for i in range(nai_distribution.shape[0]//n):
    index = i * n
    label = "{:.1E}".format(binCenters[index])
    axes[0].plot(binCenters, nai_distribution[index, :], label=label)
    
axes[0].set_title('Sensor Response to\nPhotons of Multiple Energies')
axes[0].legend()
axes[0].set_xscale('log')
axes[0].set_yscale('log')


def TGF_Reader(file):
    gammalist = np.loadtxt(file)
    TGF_energy = gammalist[0:,0] #MeV
    #TGF_energy.fill(0.662) #to simulate a monoenergetic source of Cs137
    ZenithRad = gammalist[0:,2]
    ZenithDeg = ZenithRad * 180./np.pi
    return TGF_energy, ZenithDeg

def TGF_Filter(file):
    events, degrees = TGF_Reader(file)
    #events = np.delete(events,np.where(degrees<160.))
    #events = events[events > 0.005] #cuts out energies less than 10keV
    return events

#creates the TGF spectrum binned the same as the response matrix 
def TGF_spectrum(file):
    events = TGF_Filter(file)
    hist, binEdge = np.histogram(events, bins=bins)
    return hist
    

def ResponseSpectrum(file, matrix):
    TGF_Spectrum = TGF_spectrum(file)
    #TGF_diff = 1/binCenters * np.exp(-binCenters/7.3)
    #TGF_Spectrum = TGF_diff*binWidths
    output = TGF_Spectrum * 0
    #pdb.set_trace()
    for i in np.arange(len(matrix)):
        output = output + (matrix[i]*TGF_Spectrum[i])
    return output

axes[1].plot(binCenters, np.sum(nai_distribution, axis=1), label='Combined Sensor Response')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].legend()

TGF_file = "Hera Geant 4 Simulations/Dwyer_REAM_files/6km_downward_TGF/joeAltdown_6.txt"
TGF_Spectrum = TGF_spectrum(TGF_file)/binWidths
NaIResponse = ResponseSpectrum(TGF_file, nai_distribution)/binWidths

axes[2].plot(binCenters, NaIResponse, label='TGF - NaI Response')
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].legend()

In [ ]:
counts = int(1E3)
trace_size = 3000
xmax_vis = 500
seed = 0
np.random.seed(seed)

print('Bins Range {} MeV - {} MeV'.format(binCenters[0], binCenters[-1]))
photon_time_indeces = np.random.randint(low=0, high=trace_size, size=counts)
energies = np.random.choice(binCenters, p=nai_spectrum_response / sum(nai_spectrum_response), size=counts)
# trace = np.zeros(trace_size)
# trace[photon_time_indeces] = energies_list
# Dont use arrays! just lists! to avoid zeros in spectrum hist
# On the other hand, maybe the spectrum and countrate are implicitly tied together? especailly at high count rate and coincidence
# but we should measure them seperately...

plt.tight_layout()
fig, axes = plt.subplots(1, 2, figsize=(10,3), dpi=200)
for index, energy in zip(photon_time_indeces, energies):
    axes[0].plot([index, index], [0, energy], 'r', alpha=.35)

axes[0].set_ylim([0, 1.5 * np.std(energies)])
axes[0].set_xlim([0, xmax_vis])
axes[0].set_ylabel('MeV')

axes[1].plot(binCenters, nai_spectrum_response/np.sum(nai_spectrum_response), 'b-', label='Source NaI Spectrum Dist')
hist, _ = np.histogram(energies, bins=bins)
axes[1].scatter(binCenters, hist/np.sum(hist), marker='o', facecolors='None', color='r', label='Sampled Dist')
axes[1].set_yscale('log')
axes[1].set_xscale('log')
axes[1].set_xlabel('Energy [MeV]')
axes[1].set_ylabel('Density')
axes[1].legend()

In [ ]:
temp = nai_distribution.ravel()
nonzero = temp != 0
print('Min nonzero is {}'.format(np.min(temp[nonzero])))

In [ ]:
###################################################################################
# TODO READ: = replacing zero with 1E-8 to avoid divide by zero in pymc.Interpolate
val = 1E-16
nai_distribution[np.isnan(nai_distribution)] = val
nai_distribution[nai_distribution==0] = val

#TODO this is bad? May we should reduce the bin width in each var in
###################################################################################

# nai_distribution = np.log10(nai_distribution)
# print(energies)
# energies = np.log10(energies)
# print(energies)

In [ ]:
t0 = time.time()
with pm.Model() as model:
    P_M_given_T = []
    for i in range(nai_distribution.shape[0]):
        P_M_given_Ti = pm.Interpolated.dist(x_points=binCenters, pdf_points=nai_distribution[i,:])
        P_M_given_T.append(P_M_given_Ti)
        
    w = pm.Dirichlet('weights', a=np.ones_like(binCenters))
    like = pm.Mixture('like', w=w, comp_dists=P_M_given_T, observed=energies)
    
    trace = pm.sample(draws=int(1E4), tune=1000, init="advi", chains=8, target_accept=.8, random_seed=0)
    
# 50 counts and 1K draws took 5 minutes
# 800 counts and 10K draws took 87 minutes
# 1K counts and 1K draws took 26 minutes
# 10K counts and 1K draws took 5.2 hours

# 1K counts, 500 draws, 8 chains took 20 minutes
# 1K counts, 10K draws, 8 chains took 2.4 hours
    
t1 = time.time()
total = t1 - t0
if total > 3600:
    print('Total Time: {:.2f} Hours'.format(total/3600))
elif total > 60:
    print('Total Time: {:.1f} Minutes'.format(total/60))

# with pm.Model() as model:
#     interp_pdf = pm.Interpolated.dist(x_points=x, pdf_points=y, testval=2.5)
#     w = pm.Dirichlet('w', a=np.array([1., 1.]))
#     like = pm.Mixture('like', w=w, comp_dists=[gaus, interp_pdf], testval=2.5)
#     

In [ ]:
az.summary(trace)

In [ ]:
az.plot_trace(trace, figsize=(20,8))

In [ ]:
print(trace.posterior['weights'].data.shape) # (4,1000,129)

In [ ]:
# Reshape multiple chains together and check that its correct
working = trace.posterior['weights'].data
weight_spectrum = working.reshape((-1, working.shape[2]))
# test = np.concatenate((working[0, :,:],working[1, :,:],working[2, :,:],working[3, :,:]), axis=0) #4 chains...
# print(test.shape)
# assert np.sum(weight_spectrum - test) == 0

# Sort values for Equitailed CI calcs later
sorted_weight_spectrum = np.sort(weight_spectrum, axis=0)
print(sorted_weight_spectrum.shape)


In [ ]:
n_hist = 129
weights = np.zeros((binCenters.size, n_hist)) #[True or Weights, Measured]
for i in range(sorted_weight_spectrum.shape[1]):
    hist, _ = np.histogram(sorted_weight_spectrum[:,i], bins=bins)
    weights[i, :] = hist
    
joint_T_M = weights / np.sum(weights)

plt.figure()
n = weights.shape[0]
r = np.linspace(0, 1, n)
g = np.zeros(r.size)
b = np.linspace(1, 0, n)

for i in range(n):
    plt.plot(joint_T_M[i,:], color=(r[i],g[i],b[i]))
plt.xlim([0, 20])
plt.xlabel("T_i Weight")
plt.ylabel('T_i Sampling Density')
# plt.yscale('log')

for i in range(n):
    plt.plot(joint_T_M[i, :], color=(r[i],g[i],b[i]))
plt.xlabel("T_i Weight")
plt.ylabel('T_i Sampling Density')

# peak_index = np.argmax(joint_T_M, axis=1)
# index = np.arange(129)
# peaks = joint_T_M[index, peak_index]
# plt.figure(figsize=(6,3), dpi=200)
# plt.plot(binCenters, peaks, label="MAP Mixture Weights: P(True Energy)", zorder=2)
# plt.yscale('log')
# plt.xscale('log')

hist, _ = np.histogram(energies, bins=bins)
# plt.scatter(binCenters, hist/np.sum(hist), marker='o', facecolors='None', color='r', label='Sampled Dist', zorder=1)
# plt.plot(binCenters, NaIResponse/np.sum(NaIResponse), label='True Spectrum', alpha=.25, zorder=0)
# plt.legend()
# plt.ylim([1E-5, 1E-1])

plt.figure(figsize=(7, 4), dpi=200)
ci_95 = np.percentile(sorted_weight_spectrum, [2.5, 97.5], axis=0)
ci_90 = np.percentile(sorted_weight_spectrum, [5, 95], axis=0)
ci_80 = np.percentile(sorted_weight_spectrum, [10, 90], axis=0)

weight_mean = np.mean(sorted_weight_spectrum, axis=0)
print(weight_mean.size)

# plt.plot(binCenters, peaks, label="MAP Mixture Weights", zorder=2)
# plt.scatter(binCenters, hist/np.sum(hist), marker='o', facecolors='None', color='g', label='Sampled Dist', alpha=1, zorder=1)
plt.plot(binCenters, NaIResponse/np.sum(NaIResponse), label='True Spectrum', alpha=.25, zorder=0)
plt.plot(binCenters, weight_mean, 'g', label='Posterior Mean', alpha=1, zorder=3)
plt.plot(binCenters, ci_80[0,:], 'r',  label='80% Credible Interval', alpha=.5, zorder=0)
plt.plot(binCenters, ci_80[1,:], 'r',  alpha=.5, zorder=0)
plt.plot(binCenters, ci_90[0,:], 'r',  label='90% Credible Interval', alpha=.25, zorder=0)
plt.plot(binCenters, ci_90[1,:], 'r',  alpha=.25, zorder=0)
plt.plot(binCenters, ci_95[0,:], 'r', label='95% Credible Interval', alpha=.125, zorder=0)
plt.plot(binCenters, ci_95[1,:], 'r',  alpha=.125, zorder=0)

plt.legend()
plt.ylim([1E-5, 1E-1])
plt.yscale('log')
plt.xscale('log')
plt.savefig('CIs.png')


plt.figure(figsize=(7, 4), dpi=200)
plt.scatter(binCenters, hist/np.sum(hist), marker='o', facecolors='None', color='r', label='Sampled Dist', alpha=1, zorder=1)
plt.plot(binCenters, NaIResponse/np.sum(NaIResponse), label='True Spectrum', alpha=.25, zorder=0)
plt.plot(binCenters, weight_mean, 'g', label='Posterior Mean', alpha=1, zorder=3)

plt.legend()
plt.ylim([1E-5, 1E-1])
plt.yscale('log')
plt.xscale('log')
plt.savefig('CIs.png')


In [ ]:
n_weight_hist = 70
weights = np.zeros((binCenters.size, n_weight_hist-1)) #[True or Weights, Measured]
weight_bin_edges = np.logspace(np.min(np.log10(bins)), np.max(np.log10(bins)), n_weight_hist)
weight_bins = np.diff(weight_bin_edges) + weight_bin_edges[:-1]

print('Min/Max: {:.2E}  {:.2E}'.format(np.min(weight_bin_edges), np.max(weight_bin_edges)))
                               
for i in range(sorted_weight_spectrum.shape[1]):
    hist, _ = np.histogram(sorted_weight_spectrum[:, i], bins=weight_bin_edges)
    weights[i, :] = hist
    
joint_T_M = weights / np.sum(weights)
print(joint_T_M.shape)

plt.figure()
n = weights.shape[0]
r = np.linspace(0, 1, n)
g = np.zeros(r.size)
b = np.linspace(1, 0, n)

for i in range(n):
    plt.plot(weight_bins, joint_T_M[i,:], color=(r[i],g[i],b[i]))
# plt.xlim([0, 20])
plt.xlabel("T_i Weight")
plt.ylabel('T_i Sampling Density')
# plt.yscale('log')
plt.xscale('log')

peak_index = np.argmax(joint_T_M, axis=1)
index = np.arange(n_weight_hist-1)
peaks = []
for p, i in zip(peak_index, index):
    peaks.append(joint_T_M[p, i])
plt.figure(figsize=(6,3), dpi=200)
plt.plot(weight_bins, peaks, label="Mixture Model Weights: P(True Energy)")

print(NaIResponse.shape)
plt.plot(binCenters, NaIResponse/np.sum(NaIResponse), label='True Spectrum')

plt.yscale('log')
plt.xscale('log')
plt.legend()

In [ ]:
# n_hist = 30
# weights = np.zeros((binCenters.size, n_hist)) #[True or Weights, Measured]
# for i in range(sorted_weight_spectrum.shape[1]):
#     hist, _ = np.histogram(sorted_weight_spectrum[:,i], bins=n_hist)
#     weights[i, :] = hist
#     
# joint_T_M = weights / np.sum(weights)
# print(joint_T_M.shape)

In [ ]:
# plt.figure()
# n = weights.shape[0]
# r = np.linspace(0, 1, n)
# g = np.zeros(r.size)
# b = np.linspace(1, 0, n)
# 
# for i in range(n):
#     plt.plot(joint_T_M[i, :], color=(r[i],g[i],b[i]))
# # plt.xlim([0, 20])
# plt.xlabel("T_i Weight")
# plt.ylabel('T_i Sampling Density')
# 
# peak_index = np.argmax(joint_T_M, axis=1)
# index = np.arange(129)
# peaks = joint_T_M[index, peak_index]
# plt.figure(figsize=(6,3), dpi=200)
# plt.plot(binCenters, peaks, label="Mixture Model Wights: P(True Energy)")
# plt.yscale('log')
# plt.xscale('log')
# 
# print(NaIResponse.shape)
# plt.plot(binCenters, NaIResponse/np.sum(NaIResponse), label='True Spectrum')
# plt.legend()


In [ ]:
a = np.tile(np.arange(4), 2).reshape((2,-1))
print(a, a.shape)
takex = np.array([1, 2])
takey = np.array([0, 3])

take_flat_index = takex + (a.shape[0]-1) * takey
taken = a.ravel()[take_flat_index]
print(taken)


In [ ]:
# TODO it would be interesting if there was a way to artificially increase the sampling at high energy, low probabilities.
# this idea stems from concern about the potentially large dynamic range of probability across the spectrum and how that might affect the number of samples
# needed to estimate spectrum

In [ ]:
print(weights)

In [ ]:
# plt.figure()
# # plt.pcolormesh(bins, bins, (joint_T_M/binWidths).T, shading=shading,cmap=cmap_color, norm=LogNorm(vmin=1E-5, vmax=1E0))
# plt.pcolormesh(bins, bins, (weights/binWidths).T, shading=shading,cmap=cmap_color, norm=LogNorm())
# cb = plt.colorbar()
# cb.ax.tick_params(labelsize=16)
# cb.set_label(label='Probability Density',size=18)
# plt.xlim(.1,35)
# plt.ylim(.1,35)
# plt.xscale('log')
# plt.yscale('log')
# plt.title('NaI Response Matrix',fontsize=20)
# plt.xlabel('True Energy (MeV)',fontsize=18)
# plt.ylabel('Measured Energy (MeV)',fontsize=18)
# plt.tick_params(labelsize=16)